In [1]:
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Số GPU         : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}        : {torch.cuda.get_device_name(i)}")
    mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  VRAM {i}       : {mem:.1f} GB")

CUDA available : True
Số GPU         : 1
  GPU 0        : NVIDIA RTX PRO 6000 Blackwell Server Edition
  VRAM 0       : 95.0 GB


In [2]:
!nvidia-smi

Tue May 26 14:28:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             47W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# ── Standard Library ──────────────────────────────────────────────────────
import os
import gc
import json
import math
import random
import shutil
import itertools
from datetime import datetime
from functools import partial
from pathlib import Path
from random import sample

# ── Third-party ───────────────────────────────────────────────────────────
import numpy as np
import polars as plx
import pytorch_lightning as py_light
import torch
from torch import (
    cat, concat, cumsum, diag, exp, flip, inf,
    log, logical_and, logical_or, nn,
    sigmoid, softmax, stack, tensor, tile, topk, where,
)
from torch.nn import CrossEntropyLoss, Dropout
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
from pytorch_lightning.trainer.trainer import Trainer
from pytorch_lightning.callbacks import (
    Callback, EarlyStopping, ModelCheckpoint,
)
from pytorch_lightning.loggers import CSVLogger

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# ── [OPT] Kích hoạt GPU engine cho Polars nếu có RAPIDS ──────────────────
# Polars sẽ tự dùng cuDF để xử lý data trên GPU T4, không đổi code gì khác
try:
    import cudf_polars
    print("cudf_polars loaded — Polars sẽ chạy trên GPU!")
except ImportError:
    print("cudf_polars không có — Polars chạy trên CPU (bình thường).")


Device: cuda
cudf_polars loaded — Polars sẽ chạy trên GPU!


In [4]:
DATA_PATH    = Path("/kaggle/input/datasets/trungnguyen2710t/otto-ptit-filter")
TRAIN_PARQUET = DATA_PATH / "train.parquet"
VAL_PARQUET   = DATA_PATH / "val.parquet"
TEST_PARQUET  = DATA_PATH / "test.parquet"

if not TRAIN_PARQUET.exists() or not VAL_PARQUET.exists():
    raise FileNotFoundError(f"Missing files in {DATA_PATH}")

print(f"Train : {TRAIN_PARQUET}")
print(f"Val   : {VAL_PARQUET}")


Train : /kaggle/input/datasets/trungnguyen2710t/otto-ptit-filter/train.parquet
Val   : /kaggle/input/datasets/trungnguyen2710t/otto-ptit-filter/val.parquet


In [5]:
# ── Helper: collect() dùng GPU nếu có RAPIDS, fallback CPU ───────────────
def _collect(lazy_frame):
    try:
        return lazy_frame.collect(engine="gpu")
    except Exception:
        return lazy_frame.collect()

In [6]:
# ── TRAIN: Chunk → ghi disk → sink → lazy extract ────────────────────────
def process_train_data_chunked(parquet_path, chunk_size=500_000):
    import os, shutil

    TMP_DIR   = "/kaggle/working/_tmp_train"
    FINAL_PATH = "/kaggle/working/train_processed.parquet"
    os.makedirs(TMP_DIR, exist_ok=True)

    print(f"[Train] chunk_size={chunk_size:,} sessions")
    lf = plx.scan_parquet(str(parquet_path)).select(["session", "aid", "ts", "type"])


    # ── Bước 1: itemnum ───────────────────────────────────────────────────
    # aid bắt đầu từ 0, sau khi shift +1 thì aid_max+1 = itemnum thật
    # +1 thêm 1 lần nữa để Embedding có slot riêng cho padding index 0
    # VD: aid_max=1,855,602 → sau shift max=1,855,603 → itemnum=1,855,603
    #     Embedding(itemnum+1=1,855,604): index 0=padding, 1..1,855,603=items
    itemnum = int(_collect(lf.select(plx.col("aid").max()))["aid"][0]) + 1
    # FIX: +1 shift (aid 0→1) nên itemnum cũng tăng thêm 1
    # itemnum đã = aid_max+1, sau shift aid_max → aid_max+1, nên itemnum đúng rồi
    print(f"[Train] itemnum = {itemnum}")

    # ── Bước 2: Danh sách session IDs ─────────────────────────────────────
    all_sessions = _collect(lf.select("session").unique())["session"].to_list()
    total    = len(all_sessions)
    n_chunks = (total + chunk_size - 1) // chunk_size
    print(f"[Train] Total sessions = {total:,} → {n_chunks} chunks")

    # ── Bước 3: Xử lý từng chunk → ghi disk ngay → RAM về 0 ──────────────
    for i in range(0, total, chunk_size):
        chunk_idx     = i // chunk_size + 1
        chunk_sessions = all_sessions[i : i + chunk_size]

        chunk_result = _collect(
            lf.filter(plx.col("session").is_in(chunk_sessions))
            .sort(["session", "ts"])
            .group_by("session")
            .agg([
                # FIX: shift aid+1 ngay tại đây
                # → index 0 luôn là padding, không bao giờ bị nhầm với item thật
                (plx.col("aid") + 1).sort_by(plx.col("ts")).alias("items"),
                plx.col("type").sort_by(plx.col("ts")).alias("types"),
                plx.col("ts").sort_by(plx.col("ts")).alias("timestamps"),
                # plx.col("type").filter(plx.col("type") == "orders").count().alias("order_count"),
                # plx.col("aid").count().alias("session_len"),
            ])
            # .filter(
            #     (plx.col("order_count") > 0) &
            #     (plx.col("session_len") > 1)
            # )
            .select(["session", "items", "types", "timestamps"])
        )

        path = f"{TMP_DIR}/chunk_{chunk_idx:03d}.parquet"
        chunk_result.write_parquet(path)
        n_kept = chunk_result.height
        del chunk_result
        gc.collect()
        print(f"  Chunk {chunk_idx}/{n_chunks}: {n_kept:,} sessions → disk")

    del lf
    gc.collect()

    # ── Bước 4: Gộp 13 file → 1 file duy nhất, không qua RAM ─────────────
    print("[Train] Merging chunks via sink_parquet (no RAM)...")
    plx.scan_parquet(f"{TMP_DIR}/*.parquet").sink_parquet(FINAL_PATH)
    shutil.rmtree(TMP_DIR)
    gc.collect()
    print(f"[Train] Saved → {FINAL_PATH}")

    # ── Bước 5: Lấy train_item_ids — chỉ scan 1 cột, ~50 MB ─────────────
    print("[Train] Extracting train_item_ids (lazy, 1 column)...")
    lf_final = plx.scan_parquet(FINAL_PATH)
    train_item_ids = set(
        lf_final
        .select(plx.col("items").explode())
        .unique()
        .collect()["items"]
        .to_list()
    )
    gc.collect()
    print(f"[Train] train_item_ids: {len(train_item_ids):,} unique items")

    # ── Bước 6: Lấy train_df — chỉ 2 cột, ~3-4 GB ───────────────────────
    print("[Train] Loading train_df (session, items, types, timestamps)...")
    train_df = lf_final.select(["session", "items", "types", "timestamps"]).collect()
    usernum  = train_df.height
    gc.collect()
    print(f"[Train] Final: {usernum:,} sessions")

    return train_df, usernum, itemnum, train_item_ids

In [7]:
# ── VAL: Chunk → ghi disk → sink → collect ───────────────────────────────
def process_val_data_chunked(parquet_path, train_item_ids, chunk_size=500_000):
    import os, shutil

    TMP_DIR    = "/kaggle/working/_tmp_val"
    FINAL_PATH = "/kaggle/working/val_processed.parquet"
    os.makedirs(TMP_DIR, exist_ok=True)

    print(f"[Val] chunk_size={chunk_size:,} sessions")
    lf = plx.scan_parquet(str(parquet_path)).select(["session", "aid", "ts", "type"])

    # Groupby + basic filter — shift aid+1 luôn ở đây cho gọn
    df_val = _collect(
        lf.sort(["session", "ts"])
        .group_by("session")
        .agg([
            (plx.col("aid") + 1).sort_by(plx.col("ts")).alias("items"),  # shift +1 luôn
            plx.col("type").sort_by(plx.col("ts")).alias("types"),
            plx.col("ts").sort_by(plx.col("ts")).alias("timestamps"),
        ])
        .filter(
            (plx.col("types").list.last() == "orders") &
            (plx.col("items").list.len() > 1)
        )
    )
    del lf
    gc.collect()
    print(f"[Val] After basic filter: {df_val.height:,} sessions")

    # OOV filter — items đã shift +1, train_item_ids cũng đã shift +1 → khớp trực tiếp
    train_ids_series = plx.Series("aid", list(train_item_ids), dtype=plx.Int64)
    all_sessions = df_val["session"].to_list()
    total    = len(all_sessions)
    n_chunks = (total + chunk_size - 1) // chunk_size

    for i in range(0, total, chunk_size):
        chunk_idx      = i // chunk_size + 1
        chunk_sessions = set(all_sessions[i : i + chunk_size])

        chunk = (
            df_val.filter(plx.col("session").is_in(list(chunk_sessions)))
            # OOV filter: items đã shift → so sánh thẳng với train_item_ids, không cần +1 thêm
            .filter(
                plx.col("items").list.eval(
                    plx.element().cast(plx.Int64).is_in(train_ids_series)
                ).list.all()
            )
            .with_columns([
                # context và target: items đã shift sẵn, không cần +1 thêm
                plx.col("items").list.slice(
                    0, plx.col("items").list.len() - 1
                ).alias("context"),
                plx.col("types").list.slice(
                    0, plx.col("types").list.len() - 1
                ).alias("context_types"),
                plx.col("timestamps").list.slice(
                    0, plx.col("timestamps").list.len() - 1
                ).alias("context_ts"),
                plx.col("items").list.last().alias("target"),
            ])
            .select(["session", "context", "target", "context_types", "context_ts"])
        )

        path = f"{TMP_DIR}/chunk_{chunk_idx:03d}.parquet"
        chunk.write_parquet(path)
        n_kept = chunk.height
        del chunk
        gc.collect()
        print(f"  Chunk {chunk_idx}/{n_chunks}: {n_kept:,} sessions → disk")

    del df_val
    gc.collect()

    # Gộp → 1 file, không qua RAM
    print("[Val] Merging via sink_parquet (no RAM)...")
    plx.scan_parquet(f"{TMP_DIR}/*.parquet").sink_parquet(FINAL_PATH)
    shutil.rmtree(TMP_DIR)
    gc.collect()
    print(f"[Val] Saved → {FINAL_PATH}")

    # Load — val nhỏ, OK
    print("[Val] Loading val_df...")
    df_result = plx.scan_parquet(FINAL_PATH).collect()
    gc.collect()
    print(f"[Val] Final: {df_result.height:,} sessions")
    return df_result

In [8]:
# ── Load & process data ───────────────────────────────────────────────────
CHUNK_SIZE = 500_000  # Giảm xuống 300_000 nếu vẫn OOM ở bước xử lý chunk

# Bước 1: Train
train_df, usernum, itemnum, train_item_ids = process_train_data_chunked(
    TRAIN_PARQUET, chunk_size=CHUNK_SIZE
)
gc.collect()

# Bước 2: Val
test_df = process_val_data_chunked(
    VAL_PARQUET, train_item_ids, chunk_size=CHUNK_SIZE
)
del train_item_ids
gc.collect()

num_items = itemnum
print(f"\n{'='*50}")
print(f"train sessions : {train_df['session'].n_unique():,}")
print(f"test  sessions : {test_df['session'].n_unique():,}")
print(f"num_items      : {num_items:,}")
print(f"{'='*50}")


[Train] chunk_size=500,000 sessions
[Train] itemnum = 1855603
[Train] Total sessions = 6,138,317 → 13 chunks
  Chunk 1/13: 107,152 sessions → disk
  Chunk 2/13: 106,925 sessions → disk
  Chunk 3/13: 106,469 sessions → disk
  Chunk 4/13: 107,071 sessions → disk
  Chunk 5/13: 107,082 sessions → disk
  Chunk 6/13: 107,084 sessions → disk
  Chunk 7/13: 106,775 sessions → disk
  Chunk 8/13: 107,086 sessions → disk
  Chunk 9/13: 107,257 sessions → disk
  Chunk 10/13: 107,092 sessions → disk
  Chunk 11/13: 107,313 sessions → disk
  Chunk 12/13: 106,762 sessions → disk
  Chunk 13/13: 29,524 sessions → disk
[Train] Merging chunks via sink_parquet (no RAM)...
[Train] Saved → /kaggle/working/train_processed.parquet
[Train] Extracting train_item_ids (lazy, 1 column)...
[Train] train_item_ids: 1,604,444 unique items
[Train] Loading train_df (session, items, types, timestamps)...
[Train] Final: 1,313,592 sessions
[Val] chunk_size=500,000 sessions
[Val] After basic filter: 101,226 sessions


/tmp/ipykernel_164/2876978956.py:43: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(


  Chunk 1/1: 86,727 sessions → disk
[Val] Merging via sink_parquet (no RAM)...
[Val] Saved → /kaggle/working/val_processed.parquet
[Val] Loading val_df...
[Val] Final: 86,727 sessions

train sessions : 1,313,592
test  sessions : 86,727
num_items      : 1,855,603


In [ ]:
# # Config của TRON
config = {
   "model": "sasrec",
    "dataset": "otto",
    "hidden_size": 200,
    "num_layers": 2,
    "dropout": 0.05,
    "num_batch_negatives": 127,
    "num_uniform_negatives": 8192,
    "reject_uniform_session_items": False,
    "reject_in_batch_items": True ,
    "sampling_style": "sessionwise",
    "loss": "ssm",
    "max_epochs": 50,
    "batch_size": 256,
    "max_session_length": 50,
    "lr": 0.0005,
    "limit_val_batches": 1.0,
    "accelerator": "gpu",
    "overfit_batches": 0,
    "share_embeddings": True,
    "output_bias": False,
    "shuffling_style": "no_shuffling",
    "optimizer": "adam",
    "n_behaviors": 3,
    "n_bt_buckets": 16
}

In [8]:
# #Config của SASRec
# config = {
#     "model"                      : "sasrec",
#     "dataset"                    : "otto",
#     "hidden_size"                : 200,
#     "num_layers"                 : 2,
#     "dropout"                    : 0.05,
#     "num_batch_negatives"        : 0,
#     "num_uniform_negatives"      : 1,
#     "reject_uniform_session_items": True,
#     "reject_in_batch_items"      : True,
#     "sampling_style"             : "eventwise",
#     "loss"                       : "bce",
#     "max_epochs"                 : 10,
#     "batch_size"                 : 2048,
#     "max_session_length"         : 50,
#     "lr"                         : 0.001,
#     "limit_val_batches"          : 1.0,
#     "accelerator"                : "gpu",   # T4 GPU
#     "overfit_batches"            : 0,
#     "share_embeddings"           : True,
#     "output_bias"                : False,
#     "shuffling_style"            : "no_shuffling",
#     "optimizer"                  : "adam",
# }


In [10]:
def multiply_head_with_embedding(prediction_head, embeddings):
    return prediction_head.matmul(embeddings.transpose(-1, -2))


def lookup_and_multiply(prediction_head, positives, uniform_negatives,
                        in_batch_negatives, embedding_layer, sampling_style):
    positive_logits = multiply_head_with_embedding(
        prediction_head.unsqueeze(-2),
        embedding_layer(positives).unsqueeze(-2)
    ).squeeze(-1)

    if sampling_style == "eventwise":
        uniform_negative_logits = multiply_head_with_embedding(
            prediction_head.unsqueeze(-2),
            embedding_layer(uniform_negatives)
        ).squeeze(-2)
    else:
        uniform_negative_logits = multiply_head_with_embedding(
            prediction_head, embedding_layer(uniform_negatives)
        )

    in_batch_negative_logits = multiply_head_with_embedding(
        prediction_head, embedding_layer(in_batch_negatives)
    )
    negative_logits = concat([uniform_negative_logits, in_batch_negative_logits], dim=-1)
    return positive_logits, negative_logits


ce_loss = CrossEntropyLoss(reduction="none")


def _elementwise_sampled_softmax_loss(pos_logits, neg_logits, mask, target):
    sm_logits = cat((pos_logits, neg_logits), dim=-1)
    shape = sm_logits.shape
    return ce_loss(sm_logits.reshape([-1, shape[-1]]), target).reshape([shape[0], shape[1]]) * mask


def sampled_softmax_loss(pos_logits, neg_logits, mask, device="cpu"):
    target = tensor([0], device=device).tile(mask.numel())
    return _elementwise_sampled_softmax_loss(pos_logits, neg_logits, mask, target).sum() / mask.sum()


def bce_loss(pos_logits, neg_logits, mask, epsilon=1e-10, device="cpu"):
    loss = log(1. + exp(-pos_logits) + epsilon) + log(1. + exp(neg_logits) + epsilon).mean(-1, keepdim=True)
    return (loss * mask.unsqueeze(-1)).sum() / mask.sum()


def _diff_logits(pos_logits, neg_logits):
    return pos_logits - neg_logits


def _elementwise_bpr_max_loss_per_negative(pos_logits, neg_logits):
    logits_diff = sigmoid(_diff_logits(pos_logits, neg_logits))
    s_j = softmax(neg_logits - neg_logits.max(dim=-1)[0].unsqueeze(-1), dim=-1)
    return s_j * logits_diff


def _bpr_max_loss_unregulized(pos_logits, neg_logits, mask):
    bpr_max_loss_per_element = -log(_elementwise_bpr_max_loss_per_negative(pos_logits, neg_logits).sum(dim=-1))
    return bpr_max_loss_per_element, (bpr_max_loss_per_element * mask).sum() / mask.sum()


def _bpr_max_loss_regularization(neg_logits, penalty, mask):
    regularization = penalty * (softmax(neg_logits, dim=-1) * neg_logits * neg_logits).sum(dim=-1)
    return (regularization * mask).sum() / mask.sum()


def bpr_max_loss(penalty, pos_logits, neg_logits, mask, device="cpu"):
    _, unregulized = _bpr_max_loss_unregulized(pos_logits, neg_logits, mask)
    return unregulized + _bpr_max_loss_regularization(neg_logits, penalty, mask)


def calc_loss(loss_fn, x_hat, labels, uniform_negatives, in_batch_negatives,
              mask, embeddings, sampling_style, final_activation,
              topk_sampling=False, topk_sampling_k=1000, device="cpu"):
    pos_logits, neg_logits = lookup_and_multiply(
        x_hat, labels, uniform_negatives, in_batch_negatives, embeddings, sampling_style
    )
    if topk_sampling:
        neg_logits, _ = torch.topk(neg_logits, k=topk_sampling_k, dim=-1)
    pos_scores, neg_scores = final_activation(pos_logits), final_activation(neg_logits)
    return loss_fn(pos_scores, neg_scores, mask, device=device)


In [11]:
import itertools
from random import sample

import numpy as np

def _uniform_negatives(num_items, shape):
    return np.random.randint(1, num_items + 1, shape)


def _uniform_negatives_session_rejected(num_items, shape, in_session_items):
    negatives = []
    for _ in range(np.prod(shape)):
        negative = np.random.randint(1, num_items + 1)
        while negative in in_session_items:
            negative = np.random.randint(1, num_items + 1)
        negatives.append(negative)
    return np.array(negatives).reshape(shape)


def _infer_shape(session_len, num_uniform_negatives, sampling_style):
    if sampling_style == "eventwise":
        return [session_len, num_uniform_negatives]
    elif sampling_style == "sessionwise":
        return [num_uniform_negatives]
    return []


def sample_uniform(num_items, shape, in_session_items, reject_session_items):
    if reject_session_items:
        return _uniform_negatives_session_rejected(num_items, shape, in_session_items)
    return _uniform_negatives(num_items, shape)


def sample_uniform_negatives_with_shape(clicks, num_items, session_len,
                                        num_uniform_negatives, sampling_style,
                                        reject_session_items):
    in_session_items = set(clicks)
    shape = _infer_shape(session_len, num_uniform_negatives, sampling_style)
    if shape:
        return sample_uniform(num_items, shape, in_session_items, reject_session_items)
    return np.array([])


def sample_in_batch_negatives(batch_positives, num_in_batch_negatives,
                               batch_session_len, reject_session_items):
    in_batch_negatives = []
    positive_indices = [0] + list(itertools.accumulate(batch_session_len))
    if reject_session_items:
        for i in range(len(positive_indices) - 1):
            candidates = (batch_positives[:positive_indices[i]]
                          + batch_positives[positive_indices[i + 1]:])
            in_batch_negatives.append(sample(candidates, num_in_batch_negatives))
    else:
        for _ in batch_session_len:
            in_batch_negatives.append(sample(batch_positives, num_in_batch_negatives))
    return in_batch_negatives


In [12]:
from torch import cumsum, flip, inf, stack, topk, where
def calculate_ranks(logits, labels, cutoffs):
    num_logits = logits.shape[-1]
    k = min(num_logits, cutoffs.max().item())
    _, indices = topk(logits, k=k, dim=-1)
    indices = flip(indices, dims=[-1])
    hits = indices == labels.unsqueeze(dim=-1)
    ranks = cumsum(hits, -1).sum(dim=-1) - 1.
    ranks[ranks == -1] = float('inf')
    return ranks


def pointwise_mrr(ranks, cutoffs, mask):
    res = where(ranks < cutoffs.unsqueeze(-1).unsqueeze(-1), ranks, float('inf'))
    return (1 / (res + 1)) * mask


def pointwise_recall(ranks, cutoffs, mask):
    res = ranks < cutoffs.unsqueeze(-1).unsqueeze(-1)
    return res.float() * mask


def mean_metric(pointwise_metric, mask):
    hits = pointwise_metric.sum(dim=(2, 1))
    return hits / mask.sum().clamp(0.0000005)


def validate_batch_per_timestamp(batch, x_hat, output_embedding, cut_offs):
    recalls, mrrs = [], []
    for t in range(x_hat.shape[1]):
        mask = batch['mask'][:, t]
        positives = batch['labels'][:, t]
        logits = multiply_head_with_embedding(x_hat[:, t], output_embedding.weight)
        logits[:, 0] = -inf
        ranks = calculate_ranks(logits, positives, cut_offs)
        recalls.append(pointwise_recall(ranks, cut_offs, mask).squeeze(dim=1))
        mrrs.append(pointwise_mrr(ranks, cut_offs, mask).squeeze(dim=1))
    return (mean_metric(stack(recalls, dim=2), batch["mask"]),
            mean_metric(stack(mrrs, dim=2), batch["mask"]))


In [13]:
class SasRecDataset(Dataset):

    def __init__(self, df, num_items, max_seqlen,
                 num_uniform_negatives=1, num_in_batch_negatives=0,
                 reject_uniform_session_items=False, reject_in_batch_items=True,
                 sampling_style="eventwise", shuffling_style="no_shuffling"):
        self.num_items = num_items
        self.max_seqlen = max_seqlen
        self.shuffling_style = shuffling_style
        self.num_uniform_negatives = num_uniform_negatives
        self.num_in_batch_negatives = num_in_batch_negatives
        self.reject_uniform_session_items = reject_uniform_session_items
        self.reject_in_batch_items = reject_in_batch_items
        self.sampling_style = sampling_style
        assert self.sampling_style in {"eventwise", "sessionwise", "batchwise"}
        self.session_list = df["items"].to_list()
        self.types_list = df["types"].to_list()
        self.timestamps_list = df["timestamps"].to_list()
        self.TYPE_MAP = {'clicks': 1, 'carts': 2, 'orders': 3}
        print(f"SasRecDataset: {len(self.session_list)} sessions loaded")

    def __len__(self):
        return len(self.session_list)

    def __getitem__(self, idx):
        if self.shuffling_style == "shuffle_with_replacement":
            idx = np.random.randint(0, len(self.session_list))
        clicks = [int(c) for c in self.session_list[idx]]
        types = [self.TYPE_MAP.get(str(t), 1) for t in self.types_list[idx]]
        timestamps = [int(t) for t in self.timestamps_list[idx]]
        
        clicks = clicks[-(self.max_seqlen + 1):]
        types = types[-(self.max_seqlen + 1):]
        timestamps = timestamps[-(self.max_seqlen + 1):]
        
        session_len = min(len(clicks) - 1, self.max_seqlen)
        labels = clicks[1:]
        clicks = clicks[:-1]
        b_seq = types[:-1]
        ts_seq = timestamps[:-1]
        negatives = sample_uniform_negatives_with_shape(
            clicks, self.num_items, session_len,
            self.num_uniform_negatives, self.sampling_style,
            self.reject_uniform_session_items
        )
        return {'clicks': clicks, 'labels': labels,
                'session_len': session_len, 'uniform_negatives': negatives.tolist(),
                'b_seq': b_seq, 'timestamps': ts_seq}

    def dynamic_collate(self, batch):
        max_len = self.max_seqlen
        batch_clicks, batch_mask, batch_labels = [], [], []
        batch_session_len, batch_positives = [], []
        batch_uniform_negatives, in_batch_negatives = [], []
        batch_b_seq, batch_timestamps = [], []

        for item in batch:
            sl = item["session_len"]
            pad = max_len - sl
            batch_clicks.append(pad * [0] + item["clicks"])
            batch_mask.append(pad * [0.] + sl * [1.])
            batch_labels.append(pad * [0] + item["labels"])
            batch_session_len.append(sl)
            batch_b_seq.append(pad * [0] + item["b_seq"])
            batch_timestamps.append(pad * [0] + item["timestamps"])
            batch_positives.extend(item["clicks"])

            if self.sampling_style == "eventwise":
                batch_uniform_negatives.append(
                    pad * [[0] * self.num_uniform_negatives] + item["uniform_negatives"]
                )
            elif self.sampling_style == "sessionwise":
                batch_uniform_negatives.append(item["uniform_negatives"])

        if self.sampling_style == "batchwise":
            batch_uniform_negatives = sample_uniform(
                self.num_items, [self.num_uniform_negatives],
                set(batch_positives), self.reject_in_batch_items
            )

        in_batch_negatives = sample_in_batch_negatives(
            batch_positives, self.num_in_batch_negatives,
            batch_session_len, self.reject_in_batch_items
        )

        return {
            'clicks'           : torch.tensor(batch_clicks, dtype=torch.long),
            'labels'           : torch.tensor(batch_labels, dtype=torch.long),
            'mask'             : torch.tensor(batch_mask, dtype=torch.float),
            'session_len'      : torch.tensor(batch_session_len, dtype=torch.long),
            'in_batch_negatives': torch.tensor(in_batch_negatives, dtype=torch.long),
            'uniform_negatives': torch.tensor(batch_uniform_negatives, dtype=torch.long),
            'b_seq'            : torch.tensor(batch_b_seq, dtype=torch.long),
            'timestamps'       : torch.tensor(batch_timestamps, dtype=torch.long),
        }


In [14]:
class LeaveOneOutEvalDataset(Dataset):
    def __init__(self, df, max_seqlen):
        self.max_seqlen = max_seqlen
        self.samples = []
        TYPE_MAP = {'clicks': 1, 'carts': 2, 'orders': 3}
        for row in df.iter_rows(named=True):
            context = [int(x) for x in row["context"]]
            target  = int(row["target"])
            context_types = [TYPE_MAP.get(str(t), 1) for t in row["context_types"]]
            context_ts = [int(t) for t in row["context_ts"]]
            if len(context) >= 1:
                self.samples.append({"clicks": context, "target": target, "b_seq": context_types, "timestamps": context_ts})
        print(f"LeaveOneOutEvalDataset: {len(self.samples)} samples loaded")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        clicks = s["clicks"][-self.max_seqlen:]
        b_seq = s["b_seq"][-self.max_seqlen:]
        ts_seq = s["timestamps"][-self.max_seqlen:]
        session_len = min(len(clicks), self.max_seqlen)
        
        pad = self.max_seqlen - session_len
        padded_clicks = pad * [0] + clicks
        padded_b_seq = pad * [0] + b_seq
        padded_ts_seq = pad * [0] + ts_seq
        mask = pad * [0.] + session_len * [1.]
        rated = set(clicks)
        rated.add(0)
        
        return {"clicks": padded_clicks, "mask": mask,
                "target": int(s["target"]), "rated": rated,
                "b_seq": padded_b_seq, "timestamps": padded_ts_seq}


def collate_leave_one_out_eval(batch):
    return {
        "clicks": torch.tensor([x["clicks"] for x in batch], dtype=torch.long),
        "mask"  : torch.tensor([x["mask"]   for x in batch], dtype=torch.float),
        "target": torch.tensor([x["target"] for x in batch], dtype=torch.long),
        "rated" : [x["rated"] for x in batch],
        "b_seq" : torch.tensor([x["b_seq"] for x in batch], dtype=torch.long),
        "timestamps": torch.tensor([x["timestamps"] for x in batch], dtype=torch.long),
    }


Module mới : Thêm trọng số phụ thuộc thời gian đa hành vi cho khối Attention

In [15]:
class BehaviorTemporalBias(nn.Module):
    """
    Behavior-Temporal Bias module.
    Learns a bias B[b_i, b_j, bucket(Δt)] to add to attention scores.
    """
    # Breakpoints tính bằng GIÂY (15 mốc -> 16 buckets)
    BREAKPOINTS_SEC = [1, 10, 60, 300, 1800, 7200, 21600, 43200,
                       86400, 172800, 345600, 604800, 864000, 1209600, 1814400]
    
    def __init__(self, n_behaviors=3, n_buckets=16, n_heads=1):
        super().__init__()
        self.n_b = n_behaviors + 1  # +1 cho padding (index 0)
        self.n_buckets = n_buckets
        self.n_heads = n_heads
        
        # Tensor bias learnable: shape (n_heads, N_b, N_b, N_k)
        self.bias = nn.Parameter(torch.zeros(n_heads, self.n_b, self.n_b, n_buckets))
        nn.init.normal_(self.bias, mean=0.0, std=0.02)
        
        # Đăng ký breakpoints làm buffer (tính bằng milliseconds)
        bp_ms = torch.tensor([b * 1000 for b in self.BREAKPOINTS_SEC], dtype=torch.long)
        self.register_buffer('breakpoints', bp_ms)
    
    def bucket_time(self, delta_t_ms):
        delta_t_ms = delta_t_ms.abs()
        bucket_idx = torch.bucketize(delta_t_ms, self.breakpoints, right=False)
        return bucket_idx.clamp(0, self.n_buckets - 1)
    
    def forward(self, b_seq, timestamps):
        B, T = b_seq.shape
        ts = timestamps.unsqueeze(2)     # (B, T, 1)
        delta_t = (ts - ts.transpose(1, 2)).abs()  # (B, T, T)
        
        t_bucket = self.bucket_time(delta_t)
        
        b_i = b_seq.unsqueeze(2).expand(-1, -1, T)  # query behavior
        b_j = b_seq.unsqueeze(1).expand(-1, T, -1)  # key behavior
        
        # Lấy bias cho từng head
        bias_values = self.bias[:, b_i, b_j, t_bucket]  # (H, B, T, T)
        bias_values = bias_values.permute(1, 0, 2, 3)   # (B, H, T, T)
        
        return bias_values.reshape(B * self.n_heads, T, T)



In [16]:
class DynamicPositionEmbedding(torch.nn.Module):

    def __init__(self, max_len, dimension):
        super().__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(max_len, dimension)
        self.register_buffer('pos_indices_const', torch.arange(0, max_len, dtype=torch.int))

    def forward(self, x, device='cpu'):
        seq_len = x.shape[1]
        return self.embedding(self.pos_indices_const[-seq_len:]) + x


def sample_eval_candidates(target, rated_set, num_items, n_neg=100):
    candidates = [int(target)]
    while len(candidates) < n_neg + 1:
        neg = random.randint(1, int(num_items))
        if neg not in rated_set and neg != target:
            candidates.append(neg)
    return candidates

In [17]:
class SASRec(py_light.LightningModule):

    def __init__(self, hidden_size, dropout_rate, max_len, num_items, batch_size,
                 sampling_style, topk_sampling=False, topk_sampling_k=1000,
                 learning_rate=0.001, num_layers=2, loss='bce', bpr_penalty=None,
                 optimizer='adam', output_bias=False,   share_embeddings=True,
                 final_activation=False, eval_num_negatives=99, eval_cutoff=10,
                 n_behaviors=3, n_bt_buckets=16):
        super().__init__()
        self.learning_rate = learning_rate
        self.hidden_size = hidden_size
        self.dropout_rate = dropout_rate
        self.num_items = num_items
        self.batch_size = batch_size
        self.num_layers = num_layers
        self.max_len = max_len
        self.output_bias = output_bias
        self.share_embeddings = share_embeddings
        self.register_buffer('future_mask_const',
                             torch.triu(torch.ones(max_len, max_len) * float('-inf'), diagonal=1))
        self.register_buffer('seq_diag_const',
                             ~diag(torch.ones(max_len, dtype=torch.bool)))
        self.register_buffer('bias_ones',
                             torch.ones([batch_size, max_len, 1]))

        emb_dim = hidden_size + 1 if (output_bias and share_embeddings) else hidden_size
        self.item_embedding = nn.Embedding(num_items + 1, emb_dim, padding_idx=0)
        self.positional_embedding_layer = DynamicPositionEmbedding(max_len, hidden_size)

        torch.nn.init.xavier_uniform_(self.item_embedding.weight.data)
        torch.nn.init.xavier_uniform_(self.positional_embedding_layer.embedding.weight.data)

        if share_embeddings:
            self.output_embedding = self.item_embedding
        elif output_bias:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size + 1, padding_idx=0)
        else:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)

        self.norm = nn.LayerNorm([hidden_size])
        self.input_dropout = Dropout(dropout_rate)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size, nhead=1, dim_feedforward=hidden_size,
            dropout=dropout_rate, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, norm=self.norm)
        self.merge_attn_mask = True
        self.final_activation = nn.ELU(0.5) if final_activation else nn.Identity()
        self.bt_bias = BehaviorTemporalBias(n_behaviors=n_behaviors, n_buckets=n_bt_buckets, n_heads=1)

        self.loss_fn = loss
        if loss == 'bce':
            self.loss = bce_loss
        elif loss == 'ssm':
            self.loss = sampled_softmax_loss
        elif loss == 'bpr-max':
            if bpr_penalty is None:
                raise ValueError('bpr_penalty must be provided for bpr_max loss')
            self.loss = partial(bpr_max_loss, bpr_penalty)
        else:
            raise ValueError('Loss function not supported')

        self.sampling_style = sampling_style
        self.topk_sampling = topk_sampling
        self.topk_sampling_k = topk_sampling_k
        self.optimizer = optimizer
        self.eval_num_negatives = eval_num_negatives
        self.eval_cutoff = eval_cutoff
        self.save_hyperparameters()

    def merge_attn_masks(self, padding_mask):
        batch_size = padding_mask.shape[0]
        seq_len = padding_mask.shape[1]
        # future_mask_const: 0.0 for past/current, -inf for future (upper triangle)
        future_masks = self.future_mask_const[:seq_len, :seq_len]  # (T, T)
        if not self.merge_attn_mask:
            return future_masks  # (T, T)
        
        # padding_mask: 1.0 = valid token, 0.0 = padding
        # is_pad: True where padding
        is_pad = ~padding_mask.bool()  # (B, T)
        
        # Padding keys (columns): shape (B, 1, T) -> broadcast to (B, T, T)
        # -1e9 for padding key positions, 0.0 for valid
        pad_key_mask = is_pad.unsqueeze(1).float() * -1e9
        
        # Combine: future_masks (T, T) + padding key mask (B, 1, T)
        # Broadcast: (T, T) + (B, 1, T) -> (B, T, T)
        combined = future_masks.unsqueeze(0) + pad_key_mask
        return combined

    def forward(self, item_indices, mask, b_seq=None, timestamps=None):
        att_mask = self.merge_attn_masks(mask)
        if b_seq is not None and timestamps is not None:
            bt_bias = self.bt_bias(b_seq, timestamps)
            att_mask = att_mask + bt_bias

        items = (self.item_embedding(item_indices)[:, :, :-1]
                 if self.output_bias and self.share_embeddings
                 else self.item_embedding(item_indices))
        x = items * np.sqrt(self.hidden_size)
        x = self.positional_embedding_layer(x)
        x = self.encoder(self.input_dropout(x), att_mask)
        return concat([x, self.bias_ones[:x.size(0)]], dim=-1) if self.output_bias else x

    def training_step(self, batch, _):
        x_hat = self.forward(batch["clicks"], batch["mask"], batch.get("b_seq"), batch.get("timestamps"))
        train_loss = calc_loss(
            self.loss, x_hat, batch["labels"], batch["uniform_negatives"],
            batch["in_batch_negatives"], batch["mask"], self.output_embedding,
            self.sampling_style, self.final_activation,
            self.topk_sampling, self.topk_sampling_k, self.device
        )
        self.log("train_loss", train_loss, on_step=True, on_epoch=True, prog_bar=True)
        return train_loss

    def validation_step(self, batch, _batch_idx):
        x_hat = self.forward(batch['clicks'], batch['mask'], batch.get('b_seq'), batch.get('timestamps'))
        last_hidden = x_hat[:, -1, :]
        batch_size = last_hidden.shape[0]
        hr10 = hr20 = ndcg10 = ndcg20 = 0.0

        for i in range(batch_size):
            target = int(batch['target'][i].item())
            rated  = batch['rated'][i]
            candidates = sample_eval_candidates(target, rated, self.num_items, self.eval_num_negatives)
            cand_tensor = torch.tensor(candidates, dtype=torch.long, device=self.device)
            scores = torch.matmul(last_hidden[i], self.output_embedding(cand_tensor).transpose(0, 1))
            rank = int((scores > scores[0]).to(torch.int).sum().item())
            if rank < 10:
                hr10   += 1.0
                ndcg10 += 1.0 / math.log2(rank + 2)
            if rank < 20:
                hr20   += 1.0
                ndcg20 += 1.0 / math.log2(rank + 2)

        denom = max(batch_size, 1)
        self.log('val_hr10',   hr10   / denom, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val_ndcg10', ndcg10 / denom, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val_hr20',   hr20   / denom, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val_ndcg20', ndcg20 / denom, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        if self.optimizer == 'adam':
            return torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        elif self.optimizer == 'adagrad':
            return torch.optim.Adagrad(self.parameters(), lr=self.learning_rate)
        raise ValueError('Optimizer not supported, please use adam or adagrad')

    # def export_topk_onnx(self, out_dir):
    #     TopKModel(self).export_onnx(out_dir)

    # def export(self, out_dir):
    #     self.export_topk_onnx(out_dir)

In [18]:
# class TopKModel(py_light.LightningModule):

#     def __init__(self, model: SASRec):
#         super().__init__()
#         self.model = model
#         self.example_input_array = (
#             torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]),
#             torch.tensor(10)
#         )

#     def forward(self, item_indices, k):
#         T = item_indices.shape[0]
#         mask = torch.ones(T).unsqueeze(0)                            # (1, T)
#         # BT-Bias: all items treated as 'clicks' (1), timestamps all 0
#         b_seq = torch.ones(1, T, dtype=torch.long, device=item_indices.device)
#         timestamps = torch.zeros(1, T, dtype=torch.long, device=item_indices.device)
#         self.model.merge_attn_mask = False
#         x_hat = self.model.forward(item_indices.unsqueeze(0), mask, b_seq, timestamps)[:, -1]
#         logits = multiply_head_with_embedding(x_hat, self.model.item_embedding.weight)
#         logits[0][0] = -torch.inf
#         scores, indices = torch.topk(logits, k)
#         return indices.squeeze(0), scores.squeeze(0)

#     def export_onnx(self, out_dir, verbose=True):
#         Path(out_dir).mkdir(parents=True, exist_ok=True)
#         self.to_onnx(
#             f"{out_dir}/sasrec.onnx",
#             export_params=True, opset_version=14, verbose=verbose,
#             do_constant_folding=False,
#             input_names=["item_indices", "k"],
#             output_names=["indices", "scores"],
#             dynamic_axes={
#                 'item_indices': {0: 'sequence'},
#                 'indices'     : {0: 'k'},
#                 'scores'      : {0: 'k'},
#             }
#         )


In [19]:
#Test module BehaviourTemporalBias
print("Running Smoke Test for BT-Bias...")
# 1. Test BehaviorTemporalBias module
test_bt = BehaviorTemporalBias(n_behaviors=3, n_buckets=16, n_heads=1)
sample_b_seq = torch.tensor([[1, 2, 3]])
sample_ts = torch.tensor([[1000, 5000, 100000]]) # ms
out_bias = test_bt(sample_b_seq, sample_ts)
print(f"BT-Bias output shape: {out_bias.shape} (Expected: 1, 3, 3)")
assert out_bias.shape == (1, 3, 3), "BT-Bias shape error"
print("BT-Bias module test passed!")

# 2. Test SASRec forward
sample_clicks = torch.tensor([[10, 20, 30]])
sample_mask = torch.tensor([[1., 1., 1.]])
test_model = SASRec(hidden_size=64, dropout_rate=0.1, max_len=50, num_items=1000, batch_size=1, sampling_style="sessionwise", loss="bce", n_behaviors=3, n_bt_buckets=16)
out_x = test_model(sample_clicks, sample_mask, sample_b_seq, sample_ts)
print(f"SASRec output shape: {out_x.shape} (Expected: 1, 3, 64)")
assert out_x.shape == (1, 3, 64), "SASRec output shape error"
print("SASRec forward with BT-Bias passed!\n")


Running Smoke Test for BT-Bias...
BT-Bias output shape: torch.Size([1, 3, 3]) (Expected: 1, 3, 3)
BT-Bias module test passed!
SASRec output shape: torch.Size([1, 3, 64]) (Expected: 1, 3, 64)
SASRec forward with BT-Bias passed!



/tmp/ipykernel_164/3960913828.py:46: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, norm=self.norm)


In [20]:
LOG_FILE = "/kaggle/working/training_log.txt"

class TrainingLogCallback(Callback):
    """Ghi log training/validation metrics ra file text sau mỗi epoch."""

    def __init__(self, log_path=LOG_FILE):
        super().__init__()
        self.log_path = log_path
        with open(self.log_path, "w", encoding="utf-8") as f:
            f.write("=== TRON / SASRec Training Log ===\n")
            f.write(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("-" * 70 + "\n")
            f.write(f"{'Epoch':>6}  {'Train Loss':>12}  {'HR@10':>8}  "
                    f"{'NDCG@10':>9}  {'HR@20':>8}  {'NDCG@20':>9}\n")
            f.write("-" * 70 + "\n")

    def on_train_epoch_end(self, trainer, pl_module):
        self._train_loss = float(trainer.logged_metrics.get("train_loss", float("nan")))

    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.logged_metrics
        epoch = trainer.current_epoch
        line = (f"{epoch:>6}  {getattr(self, '_train_loss', float('nan')):>12.6f}  "
                f"{float(m.get('val_hr10', float('nan'))):>8.4f}  "
                f"{float(m.get('val_ndcg10', float('nan'))):>9.4f}  "
                f"{float(m.get('val_hr20', float('nan'))):>8.4f}  "
                f"{float(m.get('val_ndcg20', float('nan'))):>9.4f}\n")
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write(line)
        print(f"[LOG] Epoch {epoch} | loss={getattr(self, '_train_loss', float('nan')):.6f} | "
              f"HR@10={float(m.get('val_hr10', 0)):.4f} NDCG@10={float(m.get('val_ndcg10', 0)):.4f} | "
              f"HR@20={float(m.get('val_hr20', 0)):.4f} NDCG@20={float(m.get('val_ndcg20', 0)):.4f}")

    def on_train_end(self, trainer, pl_module):
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write("-" * 70 + "\n")
            f.write(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            best = getattr(trainer.checkpoint_callback, 'best_model_score', None)
            if best is not None:
                f.write(f"Best val_ndcg20: {float(best):.4f}\n")
        print(f"[LOG] Training log saved → {self.log_path}")


def train_sasrec(config, train_df, test_df, num_items):
    checkpoint_callback = ModelCheckpoint(
        save_top_k=1, monitor="val_ndcg20", mode="max",
        dirpath="/kaggle/working",
        filename=f"sasrec-{config['dataset']}-" + "{epoch}-{val_ndcg20:.3f}",
    )
    early_stop = EarlyStopping(monitor="val_ndcg20", mode="max", patience=3, min_delta=0.001)
    csv_logger  = CSVLogger(save_dir="/kaggle/working/logs", name="tron")
    log_callback = TrainingLogCallback(log_path=LOG_FILE)
    trainer = Trainer(
        max_epochs=config["max_epochs"],
        precision="16-mixed",          # FIX: dùng "16-mixed" thay vì precision=16 (deprecated)
        limit_val_batches=config["limit_val_batches"],
        log_every_n_steps=1,
        accelerator=config["accelerator"],
        devices=1,
        # strategy="ddp_notebook",
        num_sanity_val_steps=1,  
        overfit_batches=config["overfit_batches"],
        callbacks=[checkpoint_callback, early_stop, log_callback],
        logger=csv_logger,
    )

    assert 0 <= config["num_batch_negatives"] < config["batch_size"]

    print("Creating train dataset...")
    train_set = SasRecDataset(
        df=train_df, num_items=num_items,
        max_seqlen=config["max_session_length"],
        num_in_batch_negatives=config["num_batch_negatives"],
        num_uniform_negatives=config["num_uniform_negatives"],
        reject_uniform_session_items=config["reject_uniform_session_items"],
        reject_in_batch_items=config["reject_in_batch_items"],
        sampling_style=config["sampling_style"],
        shuffling_style=config["shuffling_style"],
    )
    print("Creating valid dataset...")
    test_set = LeaveOneOutEvalDataset(df=test_df, max_seqlen=config["max_session_length"])

    shuffle = config["shuffling_style"] == "shuffle_without_replacement"
    train_loader = DataLoader(
        train_set, drop_last=True, batch_size=config["batch_size"],
        shuffle=shuffle, pin_memory=True, persistent_workers=True,
        num_workers=4, collate_fn=train_set.dynamic_collate,
    )
    test_loader = DataLoader(
        test_set, drop_last=False, batch_size=config["batch_size"],
        shuffle=False, pin_memory=True, persistent_workers=True,
        num_workers=4, collate_fn=collate_leave_one_out_eval,
    )

    model = SASRec(
        hidden_size=config["hidden_size"],
        dropout_rate=config["dropout"],
        max_len=config["max_session_length"],
        num_items=num_items,
        batch_size=config["batch_size"],
        sampling_style=config["sampling_style"],
        topk_sampling=config.get("topk_sampling", False),
        topk_sampling_k=config.get("topk_sampling_k", 1000),
        learning_rate=config["lr"],
        num_layers=config["num_layers"],
        loss=config["loss"],
        bpr_penalty=config.get("bpr_penalty", None),
        optimizer=config["optimizer"],
        output_bias=config["output_bias"],
        share_embeddings=config["share_embeddings"],
        eval_num_negatives=config.get("eval_num_negatives", 100),
        eval_cutoff=config.get("eval_cutoff", 10),
        n_behaviors=config.get("n_behaviors", 3),
        n_bt_buckets=config.get("n_bt_buckets", 16),
    )

    # ── In kiến trúc và tổng trọng số ──────────────────────────────────────
    sep = "=" * 70
    print(f"\n{sep}")
    print("  MÔ HÌNH: SASRec — Kiến trúc & Số tham số")
    print(sep)
    print(model)
    print(sep)

    total_params, trainable_params = 0, 0
    for p in model.parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()
    frozen_params = total_params - trainable_params

    print(f"  {'Tổng tham số':<30}: {total_params:>15,}")
    print(f"  {'Tham số có thể huấn luyện':<30}: {trainable_params:>15,}")
    print(f"  {'Tham số bị đóng băng':<30}: {frozen_params:>15,}")
    print(f"  {'Bộ nhớ ước tính (FP32)':<30}: {total_params * 4 / 1024**2:>14.2f} MB")
    print(f"  {'Bộ nhớ ước tính (FP16)':<30}: {total_params * 2 / 1024**2:>14.2f} MB")
    print(sep)

    print("\n  Chi tiết từng layer:")
    print(f"  {'Layer':<45} {'Shape':<30} {'Params':>10}")
    print("  " + "-" * 87)
    for name, param in model.named_parameters():
        shape_str = str(list(param.shape))
        print(f"  {name:<45} {shape_str:<30} {param.numel():>10,}")
    print(f"{sep}\n")
    # ───────────────────────────────────────────────────────────────────────

    return trainer, model, train_loader, test_loader

# ── Run training ────────────────────────────────────────────────────────────
trainer, model, train_loader, test_loader = train_sasrec(config, train_df, test_df, num_items)
trainer.fit(model, train_loader, val_dataloaders=test_loader)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..


Creating train dataset...
SasRecDataset: 1313592 sessions loaded
Creating valid dataset...
LeaveOneOutEvalDataset: 86727 samples loaded


/tmp/ipykernel_164/3960913828.py:46: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, norm=self.norm)
You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Server Edition') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision



  MÔ HÌNH: SASRec — Kiến trúc & Số tham số
SASRec(
  (item_embedding): Embedding(1855604, 200, padding_idx=0)
  (positional_embedding_layer): DynamicPositionEmbedding(
    (embedding): Embedding(50, 200)
  )
  (output_embedding): Embedding(1855604, 200, padding_idx=0)
  (norm): LayerNorm((200,), eps=1e-05, elementwise_affine=True)
  (input_dropout): Dropout(p=0.05, inplace=False)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=200, out_features=200, bias=True)
        )
        (linear1): Linear(in_features=200, out_features=200, bias=True)
        (dropout): Dropout(p=0.05, inplace=False)
        (linear2): Linear(in_features=200, out_features=200, bias=True)
        (norm1): LayerNorm((200,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((200,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /kaggle/working exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


...

[LOG] Epoch 0 | loss=nan | HR@10=0.9902 NDCG@10=0.9775 | HR@20=0.9939 NDCG@20=0.9785


Detected KeyboardInterrupt, attempting graceful shutdown ...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 465, in advance
    self.epoch_loop.run(self._data_fetcher)
  File

TypeError: object of type 'NoneType' has no len()

In [ ]:
# ── Lưu model sau training ────────────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working"
print("Training finished! Saving model and checkpoints...")

best_ckpt_path = trainer.checkpoint_callback.best_model_path
best_score     = trainer.checkpoint_callback.best_model_score

if best_ckpt_path and os.path.isfile(best_ckpt_path):
    print(f"Best checkpoint : {best_ckpt_path}")
    print(f"Best val_ndcg20 : {float(best_score):.4f}")

    best_model = SASRec.load_from_checkpoint(best_ckpt_path)
    best_model.eval()

    state_dict_path = os.path.join(OUTPUT_DIR, "sasrec_best_state_dict.pt")
    torch.save(best_model.state_dict(), state_dict_path)
    print(f"State dict saved: {state_dict_path}")

    config_save_path = os.path.join(OUTPUT_DIR, "sasrec_config.json")
    save_config = {**config, "num_items": int(num_items)}
    with open(config_save_path, "w", encoding="utf-8") as f:
        json.dump(save_config, f, indent=2, ensure_ascii=False)
    print(f"Config saved    : {config_save_path}")

    best_ckpt_copy = os.path.join(OUTPUT_DIR, "sasrec_best.ckpt")
    shutil.copy2(best_ckpt_path, best_ckpt_copy)
    print(f"Best .ckpt copy : {best_ckpt_copy}")

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"\nSaved files:\n")
        f.write(f"  - Best .ckpt : {best_ckpt_copy}\n")
        f.write(f"  - State dict : {state_dict_path}\n")
        f.write(f"  - Config JSON: {config_save_path}\n")
else:
    print("No checkpoint found — model was not saved.")

print(f"\nDone. All outputs are in {OUTPUT_DIR}/")
print(f"Training log    : {LOG_FILE}")

Training finished! Saving model and checkpoints...
Best checkpoint : /kaggle/working/sasrec-otto-epoch=1-val_ndcg20=0.975.ckpt
Best val_ndcg20 : 0.9754


/tmp/ipykernel_57/583689199.py:67: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, norm=self.norm)


State dict saved: /kaggle/working/sasrec_best_state_dict.pt
Config saved    : /kaggle/working/sasrec_config.json
Best .ckpt copy : /kaggle/working/sasrec_best.ckpt

Done. All outputs are in /kaggle/working/
Training log    : /kaggle/working/training_log.txt


In [ ]:
#Load test_df
test_df = _collect(plx.scan_parquet(str(TEST_PARQUET)).select(["session", "aid", "ts", "type"]))
test_df=test_df.sort(["session", "ts"]).group_by("session").agg([
                (plx.col("aid")+1).sort_by(plx.col("ts")).alias("items"),
                plx.col("type").sort_by(plx.col("ts")).alias("types"),
                plx.col("ts").sort_by(plx.col("ts")).alias("timestamps")
            ]).select(["session", "items", "types", "timestamps"])
test_df.head(10)
# Bước 2: Load test data (items đã shift +1 sẵn)
# test_df = (
#     plx.scan_parquet("/kaggle/input/.../test.parquet")
#     .select(["session", "aid", "ts"])
#     .sort(["session", "ts"])
#     .group_by("session")
#     .agg((plx.col("aid") + 1).sort_by(plx.col("ts")).alias("items"))
#     .collect()
# )


session,items
i64,list[i64]
12899779,"[59626, 875855]"
12899780,"[1142001, 582733, … 260306]"
12899781,"[141737, 199009, … 918668]"
12899782,"[1669403, 1494781, … 1738522]"
12899783,"[255298, 1114790, … 742845]"
12899784,"[1036376, 1269953, … 588459]"
12899785,"[1784452, 1169632, … 453906]"
12899786,"[955253, 955253, … 955253]"
12899787,"[1682751, 1682751, … 740563]"


In [17]:
# ── Cách load lại model ──────────────────────────────────────────────────
# Option A – đầy đủ nhất (kể cả hyperparams):
#   model = SASRec.load_from_checkpoint("/kaggle/working/sasrec_best.ckpt")
#
# Option B – nhẹ hơn (cần tạo model object trước):
#   with open("/kaggle/working/sasrec_config.json") as f:
#       cfg = json.load(f)
#   model = SASRec(hidden_size=cfg["hidden_size"], dropout_rate=cfg["dropout"],
#                  max_len=cfg["max_session_length"], num_items=cfg["num_items"],
#                  batch_size=cfg["batch_size"], sampling_style=cfg["sampling_style"],
#                  learning_rate=cfg["lr"], num_layers=cfg["num_layers"],
#                  loss=cfg["loss"], optimizer=cfg["optimizer"],
#                  output_bias=cfg["output_bias"], share_embeddings=cfg["share_embeddings"])
#   model.load_state_dict(torch.load("/kaggle/working/sasrec_best_state_dict.pt"))
#   model.eval()


In [ ]:
# INFERENCE: Dự đoán item tiếp theo cho tập Test — BATCH VERSION
# ---------------------------------------------------------------------------
# Input : test_df – Polars DataFrame có cột: session (i64), items (list[i64])
#         items đã shift +1 sẵn (giống như train/val)
# Output: DataFrame với cột "session" và "top_k_preds" (list Top-K item ID gốc)
# ---------------------------------------------------------------------------

import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import numpy as np


class InferenceDataset(Dataset):
    """Dataset cho inference — pad/truncate mỗi session."""

    def __init__(self, items_list, types_list, timestamps_list, max_len):
        self.items_list = items_list
        self.types_list = types_list
        self.timestamps_list = timestamps_list
        self.max_len    = max_len
        self.TYPE_MAP = {'clicks': 1, 'carts': 2, 'orders': 3}

    def __len__(self):
        return len(self.items_list)

    def __getitem__(self, idx):
        items   = self.items_list[idx][-self.max_len:]
        types   = [self.TYPE_MAP.get(str(t), 1) for t in self.types_list[idx][-self.max_len:]]
        timestamps = self.timestamps_list[idx][-self.max_len:]
        seq_len = len(items)
        pad = self.max_len - seq_len
        padded_items  = [0] * pad + items
        padded_types  = [0] * pad + types
        padded_ts     = [0] * pad + timestamps
        return (torch.tensor(padded_items, dtype=torch.long), 
                torch.tensor(padded_types, dtype=torch.long), 
                torch.tensor(padded_ts, dtype=torch.long))


def run_inference(model, test_df, config, num_items,
                  top_k=20, device="cuda", batch_size=256):
    """
    Chạy inference theo batch — nhanh ~1000x so với loop từng session.

    Args:
        model      : SASRec model đã load checkpoint.
        test_df    : Polars DataFrame với cột 'session' và 'items'
                     (items là list ID đã shift +1 sẵn).
        config     : dict config (dùng max_session_length).
        num_items  : itemnum (đã +1 khi tính).
        top_k      : Số item dự đoán mỗi session (mặc định 20).
        device     : 'cuda' hoặc 'cpu'.
        batch_size : Số session xử lý mỗi lần (tăng nếu VRAM còn nhiều).

    Returns:
        Polars DataFrame với cột 'session' và 'top_k_preds' (ID gốc, đã trừ 1).
    """
    model = model.to(device)
    model.eval()

    max_len  = config["max_session_length"]
    sessions = test_df["session"].to_list()
    items    = [list(row) for row in test_df["items"].to_list()]
    types    = [list(row) for row in test_df["types"].to_list()]
    timestamps = [list(row) for row in test_df["timestamps"].to_list()]

    dataset = InferenceDataset(items, types, timestamps, max_len)
    loader  = DataLoader(
        dataset,
        batch_size  = batch_size,
        shuffle     = False,
        num_workers = 4,
        pin_memory  = (device == "cuda"),
    )

    # Pre-compute embedding toàn bộ items 1 lần
    with torch.no_grad():
        all_item_ids = torch.arange(1, num_items + 1,
                                    dtype=torch.long, device=device)
        all_emb = model.output_embedding(all_item_ids)   # (num_items, D)

    total_batches = len(loader)
    all_topk      = []

    print(f"[Inference] {len(sessions):,} sessions | "
          f"batch_size={batch_size} | {total_batches} batches")

    with torch.no_grad():
        pbar = tqdm(
            loader,
            total       = total_batches,
            desc        = "Inference",
            unit        = "batch",
            bar_format  = "{l_bar}{bar:30}{r_bar}",
            dynamic_ncols = True,
        )
        for seq_batch, b_seq_batch, ts_batch in pbar:
            seq_batch   = seq_batch.to(device)
            b_seq_batch = b_seq_batch.to(device)
            ts_batch    = ts_batch.to(device)
            mask        = (seq_batch != 0).float()

            x_hat       = model.forward(seq_batch, mask, b_seq_batch, ts_batch)
            last_hidden = x_hat[:, -1, :]                  # (B, D)

            # (B, D) × (D, num_items) = (B, num_items)
            scores = torch.matmul(last_hidden, all_emb.T)

            _, top_indices = torch.topk(scores, k=top_k, dim=-1)  # (B, top_k)
            top_ids = all_item_ids[top_indices].cpu().numpy() - 1  # Trừ 1 để về ID gốc
            all_topk.append(top_ids)

            # Cập nhật thông tin trên thanh tiến độ
            done = len(all_topk) * batch_size
            pbar.set_postfix({
                "sessions": f"{min(done, len(sessions)):,}/{len(sessions):,}",
                "device"  : device,
            })

    all_topk = np.concatenate(all_topk, axis=0)   # (N, top_k)

    pred_df = plx.DataFrame({
        "session"    : sessions,
        "top_k_preds": all_topk.tolist(),
    })

    print(f"[Inference] Done — {len(sessions):,} sessions predicted")
    return pred_df


def create_submission(pred_df, output_path="/kaggle/working/submission.csv"):
    """
    Tạo file submission theo định dạng Kaggle OTTO:
        session_type          | labels
        12899779_clicks       | 129004 126836 118524 ...
        12899779_carts        | 129004 126836 118524 ...
        12899779_orders       | 129004 126836 118524 ...
    """
    submission_df = (
        pred_df
        .with_columns(
            plx.col("top_k_preds")
               .list.eval(plx.element().cast(plx.String))
               .list.join(" ")
               .alias("labels")
        )
        .select(["session", "labels"])
    )

    rows = []
    for row in tqdm(submission_df.iter_rows(named=True),
                    total=len(submission_df),
                    desc="Building submission",
                    unit="session"):
        session    = row["session"]
        labels_str = row["labels"]
        for action_type in ["clicks", "carts", "orders"]:
            rows.append({
                "session_type": f"{session}_{action_type}",
                "labels"      : labels_str,
            })

    result_df = plx.DataFrame(rows)
    result_df.write_csv(output_path)

    print(f"[Submission] Saved → {output_path}")
    print(f"  Rows   : {len(result_df):,}")
    print(f"  Preview:")
    print(result_df.head(6))
    return result_df

In [ ]:
# ── Chạy Inference + Tạo Submission ─────────────────────────────────────────

# Bước 1: Load model tốt nhất
model = SASRec.load_from_checkpoint("/kaggle/working/sasrec_best.ckpt")
model.eval()

# Bước 3: Chạy inference theo batch
pred_df = run_inference(
    model      = model,
    test_df    = test_df,
    config     = config,
    num_items  = num_items,
    top_k      = 20,
    device     = "cuda" if torch.cuda.is_available() else "cpu",
    batch_size = 256,   # tăng lên 4096 nếu VRAM còn nhiều
)

# Bước 4: Tạo submission
submission = create_submission(
    pred_df     = pred_df,
    output_path = "/kaggle/working/sasrec_submission.csv",
)

/tmp/ipykernel_57/583689199.py:67: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, norm=self.norm)


[Inference] 1,671,803 sessions | batch_size=256 | 6531 batches


Inference: 100%|██████████████████████████████| 6531/6531 [15:15<00:00,  7.13batch/s, sessions=1,671,803/1,671,803, device=cuda]


[Inference] Done — 1,671,803 sessions predicted


Building submission: 100%|██████████| 1671803/1671803 [00:03<00:00, 449903.00session/s]


[Submission] Saved → /kaggle/working/sasrec_submission.csv
  Rows   : 5,015,409
  Preview:
shape: (6, 2)
┌─────────────────┬─────────────────────────────────┐
│ session_type    ┆ labels                          │
│ ---             ┆ ---                             │
│ str             ┆ str                             │
╞═════════════════╪═════════════════════════════════╡
│ 12899779_clicks ┆ 1336956 1689538 1000721 166811… │
│ 12899779_carts  ┆ 1336956 1689538 1000721 166811… │
│ 12899779_orders ┆ 1336956 1689538 1000721 166811… │
│ 12899780_clicks ┆ 260305 1607852 1194643 1804260… │
│ 12899780_carts  ┆ 260305 1607852 1194643 1804260… │
│ 12899780_orders ┆ 260305 1607852 1194643 1804260… │
└─────────────────┴─────────────────────────────────┘


In [ ]:
import matplotlib.pyplot as plt

def visualize_bt_bias(model):
    bias = model.bt_bias.bias.detach().cpu().numpy()  # (1, 4, 4, 16)
    labels = ['pad', 'click', 'cart', 'order']
    bucket_labels = ['0s', '<10s', '<1m', '<5m', '<30m', '<2h', '<6h', '<12h',
                     '<1d', '<2d', '<4d', '<1w', '<10d', '<2w', '<3w', '>3w']

    fig, axes = plt.subplots(4, 4, figsize=(20, 16))
    for i in range(4):
        for j in range(4):
            axes[i][j].bar(range(16), bias[0, i, j, :])
            axes[i][j].set_title(f'{labels[i]} -> {labels[j]}')
            axes[i][j].set_xticks(range(16))
            axes[i][j].set_xticklabels(bucket_labels, rotation=45, fontsize=7)
    plt.tight_layout()
    plt.savefig('/kaggle/working/bt_bias_visualization.png', dpi=150)
    plt.show()

# visualize_bt_bias(model)  # Bỏ comment để chạy sau khi train
